# Baseline Model Prediksi Kerusakan PART 30 Hari - OMEXP

Notebook ini melatih dan membandingkan tiga model pada fitur inti (baseline)
yang sudah disiapkan di `analytics.failure_30d_baseline_features`: model
pembanding (dummy), Logistic Regression, dan CatBoost. Split waktu memakai
`analytics.failure_30d_model_labels` (train 2014-2024, validasi 2025, test
2026) yang sudah ada embargo 30 hari, sehingga tidak perlu split ulang secara
manual.

Fitur tambahan (challenger: lokasi, hierarki TERMINAL, interaksi) sengaja
belum dipakai di notebook ini. Nilai tambahnya baru diuji terpisah lewat
ablation study setelah baseline ini punya angka pembanding yang jelas.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve, roc_curve,
)
from catboost import CatBoostClassifier, Pool

PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 0. Tujuan dan pertanyaan yang dijawab

**Tujuan:** membangun patokan awal (baseline) yang jujur sebelum mengejar
model yang lebih rumit, dan menentukan model mana yang layak dilanjutkan.

Pertanyaan yang dijawab:

1. Seberapa jauh model sungguhan mengalahkan tebakan polos (dummy)?
2. Antara Logistic Regression (mudah dijelaskan) dan CatBoost (lebih fleksibel), mana yang lebih baik untuk data ini?
3. Kalau tim maintenance cuma sanggup memeriksa sejumlah PART tertentu, berapa banyak kerusakan yang berhasil ditemukan lebih dulu?
4. Apakah probabilitas yang dikeluarkan model bisa dipercaya (kalibrasi)?
5. Model mana yang direkomendasikan untuk dilanjutkan ke tahap berikutnya (ablation study fitur tambahan)?

Karena target sangat timpang (kerusakan cuma sebagian kecil dari semua
snapshot), **accuracy tidak dipakai sama sekali** di notebook ini - fokus ke
PR-AUC, ROC-AUC, recall/precision pada kelompok risiko tertinggi, dan
kalibrasi.


## 1. Data dan pembagian waktu

Fitur diambil dari `analytics.failure_30d_baseline_features` (16 fitur inti,
sudah lolos audit missing/redundansi/leakage). Label dan pembagian waktu
diambil terpisah dari `analytics.failure_30d_model_labels` supaya kolom masa
depan tidak pernah tergabung ke tabel fitur.

Pembagian yang dipakai (sudah termasuk embargo 30 hari di batas tahun, jadi
snapshot yang jendela 30-harinya menyeberangi batas split otomatis
dikeluarkan):

- **Train**: 2014-2024
- **Validasi**: 2025 (dipakai memilih model dan threshold)
- **Test**: 2026 (dilihat sekali di akhir, sebagai konfirmasi - 2026 baru
  separuh tahun sampai cutoff data, jadi angkanya belum final)


In [ ]:
dataset = query("""
    SELECT f.*, l.target_failure_30d, l.temporal_split
    FROM analytics.failure_30d_baseline_features f
    JOIN analytics.failure_30d_model_labels l
      USING (installation_cycle_id, item_identifier_clean, observation_on)
    WHERE l.temporal_split IN ('TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026')
""")

categorical_features = ['part_model_category', 'client_category', 'installation_age_band']
numeric_features = [
    'log_days_since_installation', 'log_total_prior_events', 'log_prior_failure_count',
    'has_prior_failure', 'log_prior_corrective_count', 'has_prior_corrective',
    'log_days_since_last_corrective', 'log_prior_distinct_places', 'log_prior_corrective_30d',
    'log_prior_failure_365d', 'log_prior_events_180d', 'month_sin', 'month_cos',
]
feature_columns = categorical_features + numeric_features
dataset[categorical_features] = dataset[categorical_features].astype(str)
dataset[numeric_features] = dataset[numeric_features].apply(pd.to_numeric)
dataset['target_failure_30d'] = dataset['target_failure_30d'].astype(bool)

splits = {}
for split_name in ['TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026']:
    part = dataset.loc[dataset.temporal_split.eq(split_name)]
    splits[split_name] = (part[feature_columns], part['target_failure_30d'])

split_summary = pd.DataFrame([
    {'Split': name, 'Jumlah snapshot': len(y), 'Positif': int(y.sum()),
     'Persentase positif (%)': round(100.0 * y.sum() / len(y), 4)}
    for name, (X, y) in splits.items()
])
display(split_summary)

X_train, y_train = splits['TRAIN_2014_2024']
X_val, y_val = splits['VALIDATION_2025']
X_test, y_test = splits['TEST_2026']

## 2. Model pembanding (dummy)

Model ini cuma menebak berdasarkan proporsi kelas di data train (tidak
melihat fitur sama sekali). Fungsinya sebagai patokan minimum - kalau model
sungguhan tidak jauh lebih baik dari ini, berarti ada yang salah.


In [ ]:
dummy_model = DummyClassifier(strategy='prior', random_state=RANDOM_STATE)
dummy_model.fit(X_train, y_train)
dummy_val_proba = dummy_model.predict_proba(X_val)[:, 1]
display(Markdown(f"Model dummy memberi probabilitas **{dummy_val_proba[0]:.4%}** untuk semua baris (sama dengan persentase positif di data train). PR-AUC dummy di validasi: **{average_precision_score(y_val, dummy_val_proba):.4%}** - ini setara dengan persentase positif di validasi itu sendiri, dan menjadi batas bawah yang harus dikalahkan model sungguhan."))

## 3. Logistic Regression

**Kenapa dicoba:** paling mudah dijelaskan ke tim non-teknis, dan fitur
numerik sudah ditransformasi (`log1p`, sin-cos) supaya cocok dipakai model
linear. `class_weight='balanced'` dipakai karena target sangat timpang.
Kategori (model PART, klien, kelompok umur) di-encode dengan one-hot;
fitur numerik diskalakan supaya tidak ada satu fitur yang mendominasi
semata-mata karena skalanya lebih besar.


In [ ]:
logistic_pipeline = Pipeline([
    ('prep', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numeric_features),
    ])),
    ('model', LogisticRegression(
        class_weight='balanced', max_iter=300, C=1.0, random_state=RANDOM_STATE,
    )),
])
logistic_pipeline.fit(X_train, y_train)
logistic_val_proba = logistic_pipeline.predict_proba(X_val)[:, 1]
display(Markdown(f"PR-AUC Logistic Regression di validasi: **{average_precision_score(y_val, logistic_val_proba):.4%}** (dummy: {average_precision_score(y_val, dummy_val_proba):.4%}). ROC-AUC: **{roc_auc_score(y_val, logistic_val_proba):.4f}**."))

## 4. CatBoost

**Kenapa dicoba:** menangani kategori (model PART, klien) secara langsung
tanpa one-hot manual, dan mampu menangkap hubungan yang tidak lurus antar
fitur. `auto_class_weights='Balanced'` dipakai karena alasan yang sama
seperti Logistic Regression. Validasi 2025 dipakai untuk early stopping
supaya model berhenti dilatih begitu tidak ada perbaikan lagi (mencegah
overfitting ke data train).


In [ ]:
train_pool = Pool(X_train, y_train, cat_features=categorical_features)
val_pool = Pool(X_val, y_val, cat_features=categorical_features)

catboost_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='PRAUC',
    auto_class_weights='Balanced',
    random_seed=RANDOM_STATE,
    early_stopping_rounds=50,
    verbose=100,
)
catboost_model.fit(train_pool, eval_set=val_pool)
catboost_val_proba = catboost_model.predict_proba(X_val)[:, 1]
display(Markdown(f"PR-AUC CatBoost di validasi: **{average_precision_score(y_val, catboost_val_proba):.4%}**. ROC-AUC: **{roc_auc_score(y_val, catboost_val_proba):.4f}**. Iterasi yang benar-benar dipakai (setelah early stopping): **{catboost_model.get_best_iteration()}** dari maksimum 1000."))

## 5. Perbandingan model di data validasi (2025)

Tiga hal yang dibandingkan: kemampuan umum memisahkan positif/negatif
(PR-AUC, ROC-AUC), dan yang paling relevan buat operasional - **dari
sekian PART dengan skor risiko tertinggi, berapa yang benar-benar rusak**
(precision@K) dan **berapa persen dari semua kerusakan berhasil tertangkap**
(recall@K) kalau tim cuma sanggup memeriksa K PART dengan skor tertinggi.


In [ ]:
def topk_table(y_true, y_proba, k_values):
    order = np.argsort(-y_proba)
    y_sorted = np.asarray(y_true)[order]
    total_positive = int(np.sum(y_true))
    rows = []
    for k in k_values:
        k = min(k, len(y_sorted))
        caught = int(y_sorted[:k].sum())
        rows.append({
            'K (jumlah PART diperiksa)': k,
            'Kerusakan tertangkap': caught,
            'Precision@K (%)': round(100.0 * caught / k, 2),
            'Recall@K (%)': round(100.0 * caught / total_positive, 2) if total_positive else 0.0,
        })
    return pd.DataFrame(rows)


models_val_proba = {
    'Dummy': dummy_val_proba,
    'Logistic Regression': logistic_val_proba,
    'CatBoost': catboost_val_proba,
}
comparison = pd.DataFrame([
    {
        'Model': name,
        'PR-AUC': average_precision_score(y_val, proba),
        'ROC-AUC': roc_auc_score(y_val, proba),
    }
    for name, proba in models_val_proba.items()
])
display(comparison.style.format({'PR-AUC': '{:.4%}', 'ROC-AUC': '{:.4f}'}))

k_values = [100, 500, 1000, 5000, int(len(y_val) * 0.01)]
for name, proba in models_val_proba.items():
    display(Markdown(f"**Recall/Precision@K - {name}**"))
    display(topk_table(y_val, proba, k_values))

plt.figure(figsize=(7, 6))
for name, proba in models_val_proba.items():
    precision, recall, _ = precision_recall_curve(y_val, proba)
    plt.plot(recall, precision, label=name)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve - Validasi 2025')
plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 6))
for name, proba in models_val_proba.items():
    fpr, tpr, _ = roc_curve(y_val, proba)
    plt.plot(fpr, tpr, label=name)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Tanpa skill')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve - Validasi 2025')
plt.legend(); plt.tight_layout(); plt.show()

## 6. Kalibrasi probabilitas

Kalibrasi mengecek apakah angka probabilitas dari model bisa dipercaya
apa adanya - misalnya kalau model bilang "70% berisiko", apakah dari
sekian PART yang diberi skor sekitar 70%, memang benar sekitar 70% dari
mereka yang akhirnya rusak. Model dengan `class_weight`/`auto_class_weights`
biasanya kurang terkalibrasi (skornya condong ke atas), jadi ini murni
pengecekan, bukan penentu model mana yang dipilih.


In [ ]:
plt.figure(figsize=(7, 6))
for name, proba in models_val_proba.items():
    if name == 'Dummy':
        continue
    frac_pos, mean_pred = calibration_curve(y_val, proba, n_bins=10, strategy='quantile')
    plt.plot(mean_pred, frac_pos, marker='o', label=name)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Kalibrasi sempurna')
plt.xlabel('Rata-rata probabilitas prediksi per kelompok')
plt.ylabel('Persentase benar-benar positif per kelompok')
plt.title('Kurva Kalibrasi - Validasi 2025')
plt.legend(); plt.tight_layout(); plt.show()
display(Markdown('**Catatan:** karena kedua model dilatih dengan pembobotan kelas (class weighting) untuk mengatasi data timpang, probabilitas mentahnya cenderung tidak langsung bisa dibaca sebagai persentase kejadian sungguhan. Kalau probabilitas terkalibrasi memang dibutuhkan (misalnya untuk laporan "peluang rusak sekian persen"), tambahkan langkah kalibrasi terpisah (`CalibratedClassifierCV`) setelah model dipilih.'))

## 7. Konfirmasi akhir di data test (2026)

Data 2026 baru dilihat di sini, sekali, sebagai konfirmasi akhir - bukan
untuk memilih model (pemilihan model sudah selesai di bagian 5 memakai data
validasi 2025). Catatan: 2026 baru berjalan sebagian sampai cutoff data,
jadi jumlah kerusakan yang tertangkap belum mencerminkan satu tahun penuh.


In [ ]:
logistic_test_proba = logistic_pipeline.predict_proba(X_test)[:, 1]
catboost_test_proba = catboost_model.predict_proba(X_test)[:, 1]
dummy_test_proba = dummy_model.predict_proba(X_test)[:, 1]

models_test_proba = {
    'Dummy': dummy_test_proba,
    'Logistic Regression': logistic_test_proba,
    'CatBoost': catboost_test_proba,
}
test_comparison = pd.DataFrame([
    {
        'Model': name,
        'PR-AUC': average_precision_score(y_test, proba),
        'ROC-AUC': roc_auc_score(y_test, proba),
    }
    for name, proba in models_test_proba.items()
])
display(test_comparison.style.format({'PR-AUC': '{:.4%}', 'ROC-AUC': '{:.4f}'}))

best_model_name = comparison.sort_values('PR-AUC', ascending=False).iloc[0]['Model']
display(Markdown(f"**Recall/Precision@K di test 2026 - model dengan PR-AUC validasi tertinggi ({best_model_name})**"))
display(topk_table(y_test, models_test_proba[best_model_name], [100, 500, 1000, 5000]))

## 8. Kesimpulan dan rekomendasi


In [ ]:
val_best_row = comparison.sort_values('PR-AUC', ascending=False).iloc[0]
val_dummy_row = comparison.loc[comparison.Model.eq('Dummy')].iloc[0]
lift_vs_dummy = val_best_row['PR-AUC'] / val_dummy_row['PR-AUC'] if val_dummy_row['PR-AUC'] else float('nan')
top1000 = topk_table(y_val, models_val_proba[val_best_row['Model']], [1000]).iloc[0]

display(Markdown(f"""**Ringkasan hasil baseline**

- Model terbaik di data validasi berdasarkan PR-AUC: **{val_best_row['Model']}** (PR-AUC {val_best_row['PR-AUC']:.4%}, ROC-AUC {val_best_row['ROC-AUC']:.4f}).
- Ini **{lift_vs_dummy:.1f}x lebih baik** dibanding tebakan polos (dummy) yang PR-AUC-nya {val_dummy_row['PR-AUC']:.4%}.
- Kalau tim maintenance memeriksa **1.000 PART** dengan skor risiko tertinggi dari model terbaik, sekitar **{int(top1000['Kerusakan tertangkap'])} di antaranya benar-benar rusak** ({top1000['Precision@K (%)']:.2f}% tepat sasaran), mencakup **{top1000['Recall@K (%)']:.2f}%** dari seluruh kerusakan yang ada di data validasi.
- Logistic Regression tetap berguna sebagai pembanding yang mudah dijelaskan, meskipun performanya biasanya di bawah CatBoost pada data seperti ini (banyak fitur kategorikal, kemungkinan hubungan tidak lurus).

**Rekomendasi:**

1. Lanjutkan dengan **{val_best_row['Model']}** sebagai kandidat utama, tapi jangan langsung dipakai produksi - masih perlu ablation study dan sensitivity analysis.
2. **Ablation study**: uji fitur tambahan (challenger) - lokasi, hierarki TERMINAL, interaksi model-lokasi - satu per satu memakai `analytics.failure_30d_challenger_features`, untuk memastikan penambahan fitur benar-benar menaikkan PR-AUC/recall@K, bukan cuma menambah kerumitan.
3. **Sensitivity analysis**: ulangi perbandingan ini memakai `is_strict_training_eligible` (versi label yang lebih ketat) untuk memastikan hasil di atas tidak goyah karena ketidakpastian label negatif.
4. Kalau probabilitas ingin dipakai langsung sebagai "peluang rusak sekian persen" di laporan, tambahkan kalibrasi terpisah sebelum dipakai.
5. Setelah dua langkah di atas stabil, baru pertimbangkan threshold operasional dan integrasi ke Grafana untuk pemantauan berkala."""))